# 00 — CARLA headless setup on Google Colab

CARLA is not officially supported on Colab, but the server can run headless (no display) using off-screen rendering. This notebook:

1. Downloads and extracts the CARLA prebuilt Linux server (must match the `carla` client version pinned in `requirements.txt`).
2. Launches it in the background with `-RenderOffScreen` (no X server needed).
3. Waits for the RPC port to come up and runs a 10-second smoke test (spawn one vehicle, tick, destroy).

**If step 3 fails**, the most common cause is Colab's GPU type not supporting the Vulkan renderer CARLA defaults to. Try re-running the launch cell with `-opengl` appended (see the fallback cell near the bottom) before assuming something else is wrong. A second common cause is picking a Colab runtime without a GPU at all — check *Runtime > Change runtime type > GPU* first.

Run this notebook first, once per Colab session (the server does not survive a runtime restart).

In [1]:
!nvidia-smi

Wed Sep  2 05:02:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
CARLA_VERSION = "0.9.15"
CARLA_TAR_URL = f"https://carla-releases.s3.us-east-005.backblazeb2.com/Linux/CARLA_{CARLA_VERSION}.tar.gz"  # the old tiny.carla.org short-link is dead; this is CARLA's actual release storage bucket
CARLA_DIR = "/content/carla_server"

import os
os.makedirs(CARLA_DIR, exist_ok=True)

In [ ]:
%%bash -s "$CARLA_TAR_URL" "$CARLA_DIR"
set -e
cd "$2"
if [ ! -f CarlaUE4.sh ]; then
  echo "Downloading CARLA server (this is large, ~15-20GB — grab a coffee)..."
  wget -q --show-progress -O carla.tar.gz "$1"
  tar -xzf carla.tar.gz
  rm carla.tar.gz
else
  echo "CARLA server already extracted, skipping download."
fi
ls

## Install the matching Python client + project requirements

In [ ]:
!pip install -q carla==0.9.15 gymnasium stable-baselines3 numpy pandas scipy matplotlib

## Launch the server headless (background process)

`-RenderOffScreen`: no display required. `-carla-server`: RPC server. `-nosound`: skip audio init (not available in a headless container).

Re-running this cell after a crash is safe — it kills any previous instance first.

In [ ]:
import subprocess
import time

subprocess.run(["pkill", "-f", "CarlaUE4"], check=False)
time.sleep(2)

carla_process = subprocess.Popen(
    [f"{CARLA_DIR}/CarlaUE4.sh", "-RenderOffScreen", "-carla-server", "-nosound", "-quality-level=Low"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("Launching CARLA server, pid:", carla_process.pid)
time.sleep(20)  # first boot is slow; increase if the connection cell below times out

## Fallback: if the cell above's server never accepts connections, try `-opengl`

Some Colab GPU types don't have a usable Vulkan ICD inside the container. Uncomment and run this cell INSTEAD of the one above if the smoke test below keeps timing out.

In [ ]:
# subprocess.run(["pkill", "-f", "CarlaUE4"], check=False)
# time.sleep(2)
# carla_process = subprocess.Popen(
#     [f"{CARLA_DIR}/CarlaUE4.sh", "-RenderOffScreen", "-opengl", "-carla-server", "-nosound", "-quality-level=Low"],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
# )
# print("Launching CARLA server (opengl fallback), pid:", carla_process.pid)
# time.sleep(20)

## Smoke test: connect, spawn one vehicle, tick, destroy

In [ ]:
import sys
sys.path.insert(0, "/content/fag-project/src")  # adjust if your repo lives elsewhere in Colab

import carla
from merge_sim import carla_utils

client = carla_utils.connect(timeout=30.0)
print("Server version:", client.get_server_version())

world = client.get_world()
with carla_utils.synchronous_mode(world, fixed_delta_seconds=0.05):
    spawn_points = world.get_map().get_spawn_points()
    vehicle = carla_utils.spawn_vehicle(world, spawn_points[0])
    assert vehicle is not None, "spawn failed — server may not be fully ready yet, wait longer and retry"
    world.tick()
    print("Smoke test OK — spawned", vehicle.type_id, "at", vehicle.get_location())
    vehicle.destroy()

## Calibrate the merge point (one-time, per CARLA version)

`scenario.py`'s auto-detection picks the first junction with >=2 converging driving lanes, which may not be the highway on-ramp you want. Print the candidates and their coordinates here; if the first one is wrong, hardcode the right `carla.Transform` into `MANUAL_MERGE_POINT` in `src/merge_sim/scenario.py`.

In [ ]:
from merge_sim.scenario import find_merge_point_candidates

world = carla_utils.load_world(client, "Town04")
candidates = find_merge_point_candidates(world)
for i, wp in enumerate(candidates):
    loc = wp.transform.location
    print(f"[{i}] road_id={wp.road_id} lane_id={wp.lane_id} location=({loc.x:.1f}, {loc.y:.1f}, {loc.z:.1f})")